# 02 · Busca: recall e limiar de recusa

1. Indexa `base/trechos.jsonl` (e5-small + FAISS).
2. Mede recall@1 e recall@3 nos casos `texto`, com destaque para os 8 casos de elevador (`metro_elev`).
3. Histograma do score do top-1: `texto` × `nao_consta` (casos `rota` ficam fora: vão para o GTFS).
4. Limiar que maximiza o F1 da recusa (positivo = `nao_consta`).
5. Salva `avaliacao/busca.json` e `avaliacao/limiar.png`.

In [1]:
import json, random, sys
from pathlib import Path

import numpy as np
import torch

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

from sonar_ia.busca import Buscador, MODELO

buscador = Buscador()
casos = [json.loads(l) for l in (RAIZ / 'avaliacao' / 'casos_teste.jsonl').read_text(encoding='utf-8').splitlines()]
texto = [c for c in casos if c['tipo'] == 'texto']
nao = [c for c in casos if c['tipo'] == 'nao_consta']
print('modelo:', MODELO, '| trechos indexados:', len(buscador.trechos))
print('casos texto:', len(texto), '| nao_consta:', len(nao), '| rota (fora da busca):', sum(c['tipo'] == 'rota' for c in casos))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

modelo: intfloat/multilingual-e5-small | trechos indexados: 50
casos texto: 30 | nao_consta: 15 | rota (fora da busca): 18


## 1. Recall@1 e recall@3 (casos `texto`)

Acerto = algum id de `trechos_esperados` aparece no top-k. A "fonte" de um caso é a fonte do primeiro trecho esperado.

In [2]:
import pandas as pd

linhas = []
for c in texto:
    top = buscador.buscar(c['pergunta'], k=3)
    ids = [t['id'] for t, _ in top]
    esp = set(c['trechos_esperados'])
    linhas.append({
        'id': c['id'], 'fonte': c['trechos_esperados'][0].rsplit('_', 1)[0],
        'hit1': ids[0] in esp, 'hit3': any(i in esp for i in ids),
        'top3': [(i, round(s, 3)) for i, (_, s) in zip(ids, top)], 'esperados': c['trechos_esperados'],
        'pergunta': c['pergunta'],
    })
res = pd.DataFrame(linhas)
recall1, recall3 = res.hit1.mean(), res.hit3.mean()
print(f'recall@1 = {recall1:.3f} ({res.hit1.sum()}/{len(res)})')
print(f'recall@3 = {recall3:.3f} ({res.hit3.sum()}/{len(res)})')

por_fonte = res.groupby('fonte').agg(casos=('id', 'count'), recall1=('hit1', 'mean'), recall3=('hit3', 'mean')).round(3)
print('\nPor fonte:'); print(por_fonte.to_string())

elev = res[res.fonte == 'metro_elev']
print(f'\n>>> DESTAQUE: {len(elev)} casos de elevador (metro_elev)  recall@1 = {elev.hit1.mean():.3f}  recall@3 = {elev.hit3.mean():.3f}')
resto = res[res.fonte != 'metro_elev']
print(f'>>> demais {len(resto)} casos texto          recall@1 = {resto.hit1.mean():.3f}  recall@3 = {resto.hit3.mean():.3f}')

recall@1 = 0.767 (23/30)
recall@3 = 0.967 (29/30)

Por fonte:
                casos  recall1  recall3
fonte                                  
cptm_acess          2     1.00      1.0
dec_caoguia         5     0.40      1.0
lbi                 4     0.75      1.0
lei_caoguia         1     1.00      1.0
metro_elev          8     1.00      1.0
metro_pref          1     1.00      1.0
smped_guia          4     0.75      1.0
sptrans_manual      5     0.60      0.8

>>> DESTAQUE: 8 casos de elevador (metro_elev)  recall@1 = 1.000  recall@3 = 1.000
>>> demais 22 casos texto          recall@1 = 0.682  recall@3 = 0.955


### Casos que erraram (top-3 recuperado, com score)

In [3]:
def mostrar(df):
    for r in df.itertuples():
        print(f"{r.id} [{r.fonte}] hit@1={r.hit1} hit@3={r.hit3}")
        print('   pergunta :', r.pergunta)
        print('   esperados:', r.esperados)
        print('   top-3    :', r.top3)

errou1 = res[~res.hit1]
print(f'ERRARAM NO TOP-1 ({len(errou1)}):\n')
mostrar(errou1)
print(f'\nERRARAM TAMBÉM NO TOP-3 ({(~res.hit3).sum()}):', list(res[~res.hit3].id))

ERRARAM NO TOP-1 (7):

texto_11 [dec_caoguia] hit@1=False hit@3=True
   pergunta : o motorista ou o metrô pode cobrar a mais por causa do meu cão-guia?
   esperados: ['dec_caoguia_001']
   top-3    : [('smped_guia_004', 0.882), ('dec_caoguia_001', 0.872), ('dec_caoguia_006', 0.866)]
texto_12 [dec_caoguia] hit@1=False hit@3=True
   pergunta : quais documentos o cao guia tem que ter pra andar de transporte?
   esperados: ['dec_caoguia_003', 'smped_guia_004']
   top-3    : [('sptrans_manual_015', 0.874), ('smped_guia_003', 0.869), ('smped_guia_004', 0.866)]
texto_14 [dec_caoguia] hit@1=False hit@3=True
   pergunta : onde meu cachorro guia fica sentado dentro do onibus ou do metrô?
   esperados: ['dec_caoguia_001']
   top-3    : [('smped_guia_004', 0.858), ('dec_caoguia_001', 0.849), ('sptrans_manual_015', 0.849)]
texto_15 [sptrans_manual] hit@1=False hit@3=True
   pergunta : se o motorista ver uma pessoa cega no ponto, ele é obrigado a parar?
   esperados: ['sptrans_manual_005']
   top-3 

### Elevador (`metro_elev`): todos os 8 casos

In [4]:
mostrar(elev)

texto_01 [metro_elev] hit@1=True hit@3=True
   pergunta : oi, a estaçao tucuruvi tem elevador?
   esperados: ['metro_elev_001']
   top-3    : [('metro_elev_001', 0.858), ('metro_elev_002', 0.85), ('metro_elev_006', 0.84)]
texto_02 [metro_elev] hit@1=True hit@3=True
   pergunta : a estação paraíso tem elevador nas duas linhas, a azul e a verde?
   esperados: ['metro_elev_002', 'metro_elev_003']
   top-3    : [('metro_elev_002', 0.877), ('metro_elev_001', 0.874), ('metro_elev_003', 0.873)]
texto_03 [metro_elev] hit@1=True hit@3=True
   pergunta : como eu entro na estaçao ana rosa pelo terminal urbano, tem elevador?
   esperados: ['metro_elev_002', 'metro_elev_003']
   top-3    : [('metro_elev_002', 0.878), ('metro_elev_005', 0.868), ('metro_elev_001', 0.866)]
texto_04 [metro_elev] hit@1=True hit@3=True
   pergunta : na estação armenia, pelo terminal urbano sul, é elevador ou rampa?
   esperados: ['metro_elev_001']
   top-3    : [('metro_elev_001', 0.869), ('metro_elev_006', 0.86), ('metr

## 2. Distribuição do score do top-1: `texto` × `nao_consta`

In [5]:
def top1(cs):
    return np.array([buscador.buscar(c['pergunta'], k=1)[0][1] for c in cs])

s_texto, s_nao = top1(texto), top1(nao)
print(f'texto      : n={len(s_texto)}  min={s_texto.min():.3f}  mediana={np.median(s_texto):.3f}  max={s_texto.max():.3f}')
print(f'nao_consta : n={len(s_nao)}  min={s_nao.min():.3f}  mediana={np.median(s_nao):.3f}  max={s_nao.max():.3f}')

texto      : n=30  min=0.852  mediana=0.876  max=0.921
nao_consta : n=15  min=0.822  mediana=0.855  max=0.877


## 3. Limiar que maximiza o F1 da recusa

Regra: recusar se `score_top1 < limiar`. Positivo = `nao_consta` (a recusa é a classe de interesse). Testa o ponto médio entre cada par de scores vizinhos; em empate de F1, usa o meio do intervalo de limiares empatados.

In [6]:
scores = np.concatenate([s_texto, s_nao])
rotulo = np.array([0] * len(s_texto) + [1] * len(s_nao))   # 1 = deve recusar

def metricas(lim):
    prev = scores < lim
    tp = int((prev & (rotulo == 1)).sum()); fp = int((prev & (rotulo == 0)).sum()); fn = int((~prev & (rotulo == 1)).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1, tp, fp, fn

ordenados = np.unique(scores)
candidatos = np.concatenate([[ordenados[0] - 1e-3], (ordenados[:-1] + ordenados[1:]) / 2, [ordenados[-1] + 1e-3]])
tabela = np.array([(lim, *metricas(lim)) for lim in candidatos])
f1_max = tabela[:, 3].max()
empatados = tabela[np.isclose(tabela[:, 3], f1_max)]
limiar = float((empatados[:, 0].min() + empatados[:, 0].max()) / 2)
prec, rec, f1, tp, fp, fn = metricas(limiar)
print(f'F1 máximo = {f1_max:.3f} em {len(empatados)} limiares candidatos, de {empatados[:, 0].min():.4f} a {empatados[:, 0].max():.4f}')
print(f'limiar escolhido = {limiar:.4f}')
print(f'precisão da recusa = {prec:.3f}  recall da recusa = {rec:.3f}  F1 = {f1:.3f}   (TP={tp} FP={fp} FN={fn})')
print(f'  FP = pergunta respondível que seria recusada: {fp} de {len(s_texto)}')
print(f'  FN = nao_consta que seria respondido (alucinação em potencial): {fn} de {len(s_nao)}')

F1 máximo = 0.757 em 1 limiares candidatos, de 0.8671 a 0.8671
limiar escolhido = 0.8671
precisão da recusa = 0.636  recall da recusa = 0.933  F1 = 0.757   (TP=14 FP=8 FN=1)
  FP = pergunta respondível que seria recusada: 8 de 30
  FN = nao_consta que seria respondido (alucinação em potencial): 1 de 15


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

bins = np.linspace(min(scores.min(), 0.7) - 0.01, scores.max() + 0.01, 25)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.hist(s_texto, bins=bins, alpha=0.75, label=f'texto (n={len(s_texto)})', color='#1f77b4')
ax.hist(s_nao, bins=bins, alpha=0.75, label=f'nao_consta (n={len(s_nao)})', color='#d62728')
ax.axvline(limiar, color='black', linestyle='--', label=f'limiar = {limiar:.3f}  (F1 da recusa = {f1:.2f})')
ax.set_xlabel('score do top-1 (cosseno, e5-small)'); ax.set_ylabel('nº de perguntas')
ax.set_title('Score do trecho mais parecido: perguntas respondíveis × sem resposta')
ax.legend(); fig.tight_layout()
fig.savefig(RAIZ / 'avaliacao' / 'limiar.png', dpi=130)
plt.close(fig)
print('salvo: avaliacao/limiar.png')

salvo: avaliacao/limiar.png


### Perguntas em cada lado do limiar

In [8]:
print('nao_consta com score >= limiar (passariam como respondíveis):')
for c, s in sorted(zip(nao, s_nao), key=lambda x: -x[1]):
    marca = 'PASSA ' if s >= limiar else 'recusa'
    top = buscador.buscar(c['pergunta'], k=1)[0][0]['id']
    print(f"  {marca} {s:.3f} {c['id']:<14} top1={top:<16} {c['pergunta']}")
print('\ntexto com score < limiar (seriam recusados indevidamente):')
for c, s in sorted(zip(texto, s_texto), key=lambda x: x[1])[:8]:
    marca = 'RECUSA' if s < limiar else 'passa '
    print(f"  {marca} {s:.3f} {c['id']:<10} {c['pergunta']}")

nao_consta com score >= limiar (passariam como respondíveis):
  PASSA  0.877 nao_consta_12  top1=smped_guia_004   como faço pra conseguir um cão guia, quanto custa?
  recusa 0.867 nao_consta_01  top1=metro_elev_003   a estação faria lima da linha 4 amarela tem elevador?
  recusa 0.865 nao_consta_06  top1=metro_elev_006   hoje tem elevador quebrado na estação tatuapé?
  recusa 0.864 nao_consta_04  top1=cptm_acess_001   o que tem de acessibilidade na estação interlagos da linha 9?
  recusa 0.858 nao_consta_03  top1=metro_elev_005   a estaçao carapicuiba da linha 8 diamante é acessivel pra cego?
  recusa 0.857 nao_consta_02  top1=metro_elev_002   tem elevador na estação adolfo pinheiro da linha 5 lilás?
  recusa 0.857 nao_consta_15  top1=metro_pref_001   qual o telefone da central de atendimento do metrô?
  recusa 0.855 nao_consta_09  top1=metro_elev_002   a estação carioca do metrô do rio de janeiro tem elevador?
  recusa 0.854 nao_consta_05  top1=metro_elev_002   o elevador da estaçao s

In [9]:
pega = {c['id']: (buscador.buscar(c['pergunta'], k=3), s) for c, s in zip(nao, s_nao)}
for id_ in ('nao_consta_01', 'nao_consta_14'):
    top, s = pega[id_]
    pergunta = next(c['pergunta'] for c in nao if c['id'] == id_)
    print(f"{id_}: {pergunta!r}  score top-1 = {s:.3f}  (limiar {limiar:.3f}) -> {'PASSA do limiar' if s >= limiar else 'recusada'}")
    for t, sc in top:
        print(f'    {sc:.3f} {t["id"]}')

nao_consta_01: 'a estação faria lima da linha 4 amarela tem elevador?'  score top-1 = 0.867  (limiar 0.867) -> recusada
    0.867 metro_elev_003
    0.867 metro_elev_005
    0.858 metro_elev_001
nao_consta_14: 'tem embarque preferencial na estação tucuruvi?'  score top-1 = 0.854  (limiar 0.867) -> recusada
    0.854 metro_elev_001
    0.843 metro_pref_001
    0.836 metro_elev_004


## 4. Salvar `avaliacao/busca.json`

In [10]:
# métricas extras (comparação v1 x v2): margem top1-top2 nos elevadores, pegadinhas e limiar como pré-filtro
from sonar_ia.busca import limiar_prefiltro

elev_casos = [c for c in texto if c['trechos_esperados'][0].startswith('metro_elev')]
margens = []
for c in elev_casos:
    (_, s1), (_, s2) = buscador.buscar(c['pergunta'], k=2)
    margens.append(s1 - s2)
score_nao = {c['id']: float(s) for c, s in zip(nao, s_nao)}
lim_pre = limiar_prefiltro(s_texto)
extras = {
    'margem_media_top1_top2_elevador': round(float(np.mean(margens)), 4),
    'pegadinhas': {k: round(score_nao[k], 4) for k in ('nao_consta_01', 'nao_consta_02', 'nao_consta_03')},
    'falsa_recusa_no_limiar_escolhido': int((s_texto < limiar).sum()),
    'recusa_correta_no_limiar_escolhido': int((s_nao < limiar).sum()),
    'limiar_prefiltro': lim_pre,
    'falsa_recusa_prefiltro': int((s_texto < lim_pre).sum()),
    'recusa_correta_prefiltro': int((s_nao < lim_pre).sum()),
    'recall@1_elevador': round(float(elev.hit1.mean()), 4),
}
print(json.dumps(extras, ensure_ascii=False, indent=2))

{
  "margem_media_top1_top2_elevador": 0.0108,
  "pegadinhas": {
    "nao_consta_01": 0.8667,
    "nao_consta_02": 0.8574,
    "nao_consta_03": 0.8582
  },
  "falsa_recusa_no_limiar_escolhido": 8,
  "recusa_correta_no_limiar_escolhido": 14,
  "limiar_prefiltro": 0.8537,
  "falsa_recusa_prefiltro": 1,
  "recusa_correta_prefiltro": 5,
  "recall@1_elevador": 1.0
}


In [11]:
resultado = {
    'recall@1': round(float(recall1), 4), 'recall@3': round(float(recall3), 4),
    'recall_por_fonte': {f: {'casos': int(r.casos), 'recall@1': float(r.recall1), 'recall@3': float(r.recall3)} for f, r in por_fonte.iterrows()},
    'limiar': round(limiar, 4), 'f1': round(float(f1), 4), 'precisao': round(float(prec), 4), 'recall_recusa': round(float(rec), 4),
    'modelo_embedding': MODELO,
    'extras': extras,
}
(RAIZ / 'avaliacao' / 'busca.json').write_text(json.dumps(resultado, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(resultado, ensure_ascii=False, indent=2))

{
  "recall@1": 0.7667,
  "recall@3": 0.9667,
  "recall_por_fonte": {
    "cptm_acess": {
      "casos": 2,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "dec_caoguia": {
      "casos": 5,
      "recall@1": 0.4,
      "recall@3": 1.0
    },
    "lbi": {
      "casos": 4,
      "recall@1": 0.75,
      "recall@3": 1.0
    },
    "lei_caoguia": {
      "casos": 1,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "metro_elev": {
      "casos": 8,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "metro_pref": {
      "casos": 1,
      "recall@1": 1.0,
      "recall@3": 1.0
    },
    "smped_guia": {
      "casos": 4,
      "recall@1": 0.75,
      "recall@3": 1.0
    },
    "sptrans_manual": {
      "casos": 5,
      "recall@1": 0.6,
      "recall@3": 0.8
    }
  },
  "limiar": 0.8671,
  "f1": 0.7568,
  "precisao": 0.6364,
  "recall_recusa": 0.9333,
  "modelo_embedding": "intfloat/multilingual-e5-small",
  "extras": {
    "margem_media_top1_top2_elevador": 0.0108,
   

**Nota (etapa 3, v1):** os números desta justificativa foram escritos antes de corrigir o gabarito do `texto_08` (agora aceita também `metro_elev_008`). Com a correção, o recall@1 passa de 0,733 para 0,767 e o recall@1 de elevador, de 0,875 para 1,0; o limiar e as demais conclusões seguem a saída das células acima.

## 5. Justificativa do limiar

**Escolha.** O limiar que maximiza o F1 da recusa (positivo = `nao_consta`) é **0,8671**: recusar quando o score do top-1 fica abaixo disso. Nesse ponto, **precisão = 0,636, recall = 0,933 e F1 = 0,757** (TP=14, FP=8, FN=1). Das 15 perguntas sem resposta, 14 são recusadas; das 30 respondíveis, 8 também seriam recusadas (27%).

**O limiar é frágil, e isso é um achado, não um detalhe.**
- As duas distribuições se sobrepõem quase por inteiro: `texto` vai de 0,852 a 0,921 (mediana 0,876) e `nao_consta` de 0,822 a 0,877 (mediana 0,855). A faixa de 0,852 a 0,877 tem as duas classes misturadas.
- O F1 máximo ocorre em um único ponto (0,8671), sem plateau. Qualquer deslocamento pequeno muda o resultado.
- O limiar foi escolhido nos mesmos 45 casos em que foi medido (não há conjunto separado), então 0,757 é otimista.
- O e5-small comprime os scores (todos entre 0,82 e 0,92), e a diferença entre acerto e erro é de centésimos.

**Custo do ponto escolhido.** Os 8 respondíveis recusados indevidamente são texto_09, 19, 14, 01, 16, 07, 05 e 15. Três são de elevador (01 Tucuruvi, 07 Brigadeiro e 05 Jabaquara, com scores de 0,858 a 0,863). A única alucinação em potencial (FN) é **nao_consta_12** (cão-guia: conseguir/custo), com score 0,877 e top-1 `smped_guia_004`, que fala de cão-guia: o tema casa, mesmo sem a resposta.

**Recall.** recall@1 = 0,733 (22/30) e recall@3 = 0,933 (28/30). Em `metro_elev`, recall@1 = recall@3 = 0,875 (7/8), acima dos demais casos (0,682 e 0,955). A suspeita de diluição **não se confirma pelo recall, mas aparece nos scores e nas margens**: os acertos de elevador ganham do segundo colocado por 0,001 a 0,018 (ex.: texto_02, 0,877 contra 0,874; texto_01, 0,858 contra 0,850), e as perguntas `nao_consta` sobre estações que não estão na base (01 Faria Lima: 0,867; 02 Adolfo Pinheiro: 0,857; 03 Carapicuíba: 0,858) recuperam trechos `metro_elev` com score igual ao dos acertos reais (0,858 a 0,878). O nome da estação pesa pouco no embedding de um trecho que agrupa ~15 estações. O único erro de elevador é texto_08 (Sapopemba): o top-1 é `metro_elev_008` (0,907), que também traz Sapopemba lado ímpar com a mesma resposta (rampa ou elevador). Ou seja, o gabarito do caso só lista o 007 e é discutível.

**Pegadinhas.** Nenhuma passou do limiar, mas por margens muito diferentes:
- **nao_consta_01 (Faria Lima, Linha 4):** score 0,867 contra limiar 0,8671, ou seja, **recusada por uma margem de ~0,0001**. Top-1 empatado entre `metro_elev_003` (que cita a "Linha 4-Amarela" só como conexão da Consolação) e `metro_elev_005`. Na prática, passaria com qualquer ajuste mínimo. É o caso mais perigoso (resposta plausível, com cara de fonte).
- **nao_consta_14 (embarque preferencial no Tucuruvi):** score 0,854, 0,013 abaixo do limiar. O top-1 é `metro_elev_001` (que contém "Tucuruvi", mas é sobre elevador), e `metro_pref_001`, o trecho certo, fica em segundo (0,843). A palavra "Tucuruvi" puxou a busca para a fonte errada: o gerador receberia um trecho sobre a estação sem nada sobre embarque preferencial.

**Conclusão.** O limiar 0,8671 é o melhor ponto possível com este modelo e este chunking, mas separa mal as duas classes (F1 0,757). Isso confirma a necessidade de melhorar o chunking de `metro_elev` (um trecho por estação ou por linha, com o nome da estação em destaque) antes de depender só do score para recusar.